In [ ]:
!pip install -q tensorflow pillow matplotlib scikit-learn seaborn
 
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score,
                             recall_score, f1_score)
import seaborn as sns
import json, time, random
from collections import Counter
 
print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
IMG_SIZE    = 224
BATCH_SIZE  = 64
EPOCHS      = 10
LR          = 1e-4
LABEL_SMOOTH= 0.1
TRAIN_LIMIT = 5000   # per class
VAL_LIMIT   = 1000   # per class
 
DATA_PATH  = '/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset'
print("Config ready.")

In [ ]:
def find_folder(base, names):
    for f in os.listdir(base):
        if f.lower() in [n.lower() for n in names]:
            return os.path.join(base, f)
    return None
 
TRAIN_PATH = find_folder(DATA_PATH, ['train', 'training'])
VAL_PATH   = find_folder(DATA_PATH, ['validation', 'val', 'valid', 'test'])
 
print(f"Train: {TRAIN_PATH}")
print(f"Val:   {VAL_PATH}")
 

In [ ]:
def get_image_paths_and_labels(folder, limit_per_class=None):
    class_names = sorted([
        d for d in os.listdir(folder)
        if os.path.isdir(os.path.join(folder, d))
    ])
    all_paths, all_labels = [], []
    for label, cls in enumerate(class_names):
        cls_path = os.path.join(folder, cls)
        files = [
            os.path.join(cls_path, f)
            for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        if limit_per_class:
            files = random.sample(files, min(limit_per_class, len(files)))
        all_paths.extend(files)
        all_labels.extend([label] * len(files))
        print(f"  {cls}: {len(files)} images (label={label})")
 
    combined = list(zip(all_paths, all_labels))
    random.shuffle(combined)
    paths, labels = zip(*combined)
    return list(paths), list(labels), class_names
 
print("Loading train...")
train_paths, train_labels, CLASS_NAMES = get_image_paths_and_labels(
    TRAIN_PATH, limit_per_class=TRAIN_LIMIT)
 
print("\nLoading val...")
val_paths, val_labels, _ = get_image_paths_and_labels(
    VAL_PATH, limit_per_class=VAL_LIMIT)
 
print(f"\nTotal train: {len(train_paths)}")
print(f"Total val:   {len(val_paths)}")
print(f"Classes: {CLASS_NAMES}")
 

In [ ]:
# WHY FREQUENCY DOMAIN?
# GAN-generated faces have characteristic artifacts in the
# frequency spectrum — repeating patterns invisible to the eye
# but detectable via FFT. Real faces have natural frequency
# distributions; fakes have anomalous high-frequency spikes.
 
def compute_fft_features(image_path):
    """
    Load image → convert to grayscale → 2D FFT →
    log magnitude spectrum → resize to fixed shape
    Returns: (64, 64) float32 array
    """
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.image.rgb_to_grayscale(img)          # (224,224,1)
    img = tf.squeeze(img, axis=-1)                 # (224,224)
    img = tf.cast(img, tf.float32) / 255.0
 
    # 2D FFT
    fft = tf.signal.fft2d(tf.cast(img, tf.complex64))
    fft_shifted = tf.signal.fftshift(fft)          # center low freqs
    magnitude   = tf.abs(fft_shifted)
    log_mag     = tf.math.log(magnitude + 1e-8)    # log scale
 
    # Resize to 64x64 for efficiency
    log_mag = tf.expand_dims(log_mag, axis=-1)     # (224,224,1)
    log_mag = tf.image.resize(log_mag, [64, 64])   # (64,64,1)
    log_mag = tf.squeeze(log_mag, axis=-1)          # (64,64)
 
    # Normalize to [0,1]
    mn  = tf.reduce_min(log_mag)
    mx  = tf.reduce_max(log_mag)
    log_mag = (log_mag - mn) / (mx - mn + 1e-8)
 
    return log_mag  # (64,64) float32
 
# Test on one image
test_fft = compute_fft_features(train_paths[0])
print(f"FFT feature shape: {test_fft.shape}")  # should be (64,64)

In [ ]:
print("Visualizing FFT features for Real vs Fake...")
 
# Find one real and one fake sample
real_idx = next(i for i, l in enumerate(train_labels) if l == CLASS_NAMES.index('Real'))
fake_idx = next(i for i, l in enumerate(train_labels) if l == CLASS_NAMES.index('Fake'))
 
real_fft = compute_fft_features(train_paths[real_idx]).numpy()
fake_fft = compute_fft_features(train_paths[fake_idx]).numpy()
 
def load_rgb(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return (img.numpy() / 255.0)
 
real_img = load_rgb(train_paths[real_idx])
fake_img = load_rgb(train_paths[fake_idx])
 
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0,0].imshow(real_img);      axes[0,0].set_title('Real Image');    axes[0,0].axis('off')
axes[0,1].imshow(real_fft, cmap='hot'); axes[0,1].set_title('Real FFT Spectrum'); axes[0,1].axis('off')
axes[1,0].imshow(fake_img);      axes[1,0].set_title('Fake Image');    axes[1,0].axis('off')
axes[1,1].imshow(fake_fft, cmap='hot'); axes[1,1].set_title('Fake FFT Spectrum'); axes[1,1].axis('off')
 
plt.suptitle('Spatial vs Frequency Domain: Real vs Fake', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/fft_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fft_visualization.png")

In [ ]:
# Each sample = (spatial_image, fft_features), label
# spatial: (224,224,3)  → EfficientNetB4 branch
# fft:     (64,64,1)    → CNN branch
 
def load_sample(path, label):
    # --- Spatial branch ---
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = preprocess_input(img)                    # EfficientNet preprocessing
 
    # --- Frequency branch ---
    gray = tf.image.rgb_to_grayscale(
        tf.image.resize(
            tf.cast(tf.image.decode_jpeg(
                tf.io.read_file(path), channels=3), tf.float32),
            [IMG_SIZE, IMG_SIZE]
        )
    )
    gray   = tf.squeeze(gray, axis=-1) / 255.0
    fft    = tf.signal.fft2d(tf.cast(gray, tf.complex64))
    fft_s  = tf.signal.fftshift(fft)
    mag    = tf.math.log(tf.abs(fft_s) + 1e-8)
    mag    = tf.expand_dims(mag, axis=-1)
    mag    = tf.image.resize(mag, [64, 64])        # (64,64,1)
    mn, mx = tf.reduce_min(mag), tf.reduce_max(mag)
    mag    = (mag - mn) / (mx - mn + 1e-8)
 
    return (img, mag), label
 
def augment_sample(inputs, label):
    img, mag = inputs
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    return (img, mag), label
 
AUTOTUNE = tf.data.AUTOTUNE
 
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = (train_ds
    .shuffle(len(train_paths), seed=42)
    .map(load_sample,    num_parallel_calls=AUTOTUNE)
    .map(augment_sample, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
 
val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = (val_ds
    .map(load_sample, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
 
print(f"Steps/epoch: {len(train_paths) // BATCH_SIZE}")
print(f"Val steps:   {len(val_paths) // BATCH_SIZE}")
 
# Verify batch shapes
for (imgs, ffts), lbls in train_ds.take(1):
    print(f"Spatial batch: {imgs.shape}")   # (64, 224, 224, 3)
    print(f"FFT batch:     {ffts.shape}")   # (64, 64, 64, 1)
    print(f"Labels:        {lbls.shape}")   # (64,)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
 
cw = compute_class_weight('balanced',
                           classes=np.unique(train_labels),
                           y=train_labels)
class_weight_dict = dict(enumerate(cw))
print(f"Class weights: {class_weight_dict}")

In [ ]:
# Branch 1: EfficientNetB4 on spatial image
# Branch 2: Small CNN on FFT magnitude spectrum
# Fusion: concatenate → dense head → sigmoid
 
def build_dual_model():
    # --- Spatial Branch (EfficientNetB4) ---
    base = EfficientNetB4(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False  # frozen for Phase 1
 
    spatial_input = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='spatial')
    x1 = base(spatial_input, training=False)
    x1 = layers.GlobalAveragePooling2D()(x1)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Dropout(0.4)(x1)
    x1 = layers.Dense(256, activation='relu',
                      kernel_regularizer=keras.regularizers.l2(1e-4))(x1)
 
    # --- Frequency Branch (lightweight CNN) ---
    fft_input = keras.Input(shape=(64, 64, 1), name='frequency')
    x2 = layers.Conv2D(32, 3, activation='relu', padding='same')(fft_input)
    x2 = layers.BatchNormalization()(x2)
    x2 = layers.MaxPooling2D(2)(x2)                    # 32x32
 
    x2 = layers.Conv2D(64, 3, activation='relu', padding='same')(x2)
    x2 = layers.BatchNormalization()(x2)
    x2 = layers.MaxPooling2D(2)(x2)                    # 16x16
 
    x2 = layers.Conv2D(128, 3, activation='relu', padding='same')(x2)
    x2 = layers.BatchNormalization()(x2)
    x2 = layers.GlobalAveragePooling2D()(x2)
    x2 = layers.Dropout(0.3)(x2)
    x2 = layers.Dense(128, activation='relu')(x2)
 
    # --- Fusion ---
    fused  = layers.Concatenate()([x1, x2])            # 256+128=384
    fused  = layers.BatchNormalization()(fused)
    fused  = layers.Dropout(0.4)(fused)
    fused  = layers.Dense(128, activation='relu')(fused)
    fused  = layers.Dropout(0.3)(fused)
    output = layers.Dense(1, activation='sigmoid', name='output')(fused)
 
    model = keras.Model(
        inputs=[spatial_input, fft_input],
        outputs=output
    )
    return model, base
 
model, base_model = build_dual_model()
 
model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision', thresholds=0.5),
        keras.metrics.Recall(name='recall', thresholds=0.5),
        keras.metrics.AUC(name='auc')
    ]
)
 
model.summary()
print(f"\nTotal params: {model.count_params():,}")

In [ ]:
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
 
def make_callbacks(prefix):
    return [
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=4,
            restore_best_weights=True, mode='max', verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=2, min_lr=1e-7, verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            f'/kaggle/working/checkpoints/best_{prefix}.keras',
            monitor='val_accuracy', save_best_only=True,
            mode='max', verbose=1
        )
    ]
 
print("Callbacks ready.")

In [ ]:
print("=" * 55)
print("PHASE 1: Spatial + Frequency heads (EfficientNet frozen)")
print("=" * 55)
 
start = time.time()
history1 = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=make_callbacks('phase1'),
    verbose=1
)
print(f"\nPhase 1 done in {(time.time()-start)/60:.1f} min")

In [ ]:
print("\n" + "=" * 55)
print("PHASE 2: Fine-tuning top 30 layers of EfficientNetB4")
print("=" * 55)
 
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
 
model.compile(
    optimizer=keras.optimizers.Adam(LR / 10),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision', thresholds=0.5),
        keras.metrics.Recall(name='recall', thresholds=0.5),
        keras.metrics.AUC(name='auc')
    ]
)
 
start = time.time()
history2 = model.fit(
    train_ds,
    epochs=8,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=make_callbacks('phase2'),
    verbose=1
)
print(f"\nPhase 2 done in {(time.time()-start)/60:.1f} min")

In [ ]:
print("\nEvaluating...")
all_preds, all_true = [], []
 
for (imgs, ffts), lbls in val_ds:
    preds = model.predict([imgs, ffts], verbose=0)
    all_preds.extend(preds.flatten())
    all_true.extend(lbls.numpy())
 
pred_classes = (np.array(all_preds) > 0.5).astype(int)
true_classes  = np.array(all_true)
 
acc  = accuracy_score(true_classes, pred_classes)
prec = precision_score(true_classes, pred_classes, zero_division=0)
rec  = recall_score(true_classes, pred_classes, zero_division=0)
f1   = f1_score(true_classes, pred_classes, zero_division=0)
 
print("\n" + "=" * 55)
print("STEP 2 FINAL RESULTS")
print("=" * 55)
print(f"Accuracy:  {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall:    {rec*100:.2f}%")
print(f"F1-Score:  {f1*100:.2f}%")
print("=" * 55)
print(classification_report(true_classes, pred_classes,
      target_names=CLASS_NAMES))


In [ ]:
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history.get(key, [])
    return combined
 
hist   = combine_histories(history1, history2)
p1_end = len(history1.history['accuracy'])
 
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Dual-Branch Model (Spatial + Frequency) — Deepfake Detection', fontsize=13)
 
for ax, metric, title in [
    (axes[0,0], 'accuracy',  'Accuracy'),
    (axes[0,1], 'loss',      'Loss'),
    (axes[1,0], 'precision', 'Precision'),
    (axes[1,1], 'recall',    'Recall'),
]:
    ax.plot(hist[metric],          'o-', label='Train', lw=2)
    ax.plot(hist[f'val_{metric}'], 's-', label='Val',   lw=2)
    ax.axvline(p1_end - 1, color='gray', linestyle='--',
               alpha=0.6, label='Fine-tune starts')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
 
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves_step2.png', dpi=150, bbox_inches='tight')
plt.show()
 
# Confusion matrix
cm = confusion_matrix(true_classes, pred_classes)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={'size': 16})
plt.title('Confusion Matrix — Dual Branch', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix_step2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
model.save('/kaggle/working/deepfake_dual_branch_final.keras')
 
serializable = {k: [float(v) for v in vals] for k, vals in hist.items()}
with open('/kaggle/working/training_history_step2.json', 'w') as f:
    json.dump(serializable, f)
 
print("\n" + "=" * 55)
print("FILES SAVED TO /kaggle/working/")
print("  - deepfake_dual_branch_final.keras")
print("  - checkpoints/best_phase1.keras")
print("  - checkpoints/best_phase2.keras")
print("  - training_curves_step2.png")
print("  - confusion_matrix_step2.png")
print("  - fft_visualization.png")
print("  - training_history_step2.json")
print("=" * 55)
print("\nStep 2 DONE. Next → Step 3: GradCAM Explainability")